# 03 — Train Orthoptera Classifier
Fine-tune a CNN on the prepared label splits using OpenSoundscape, then export the trained model to `models/` for use in BASE.

**Kernel:** `Python (orthoptera-training)`  
**Prerequisites:** Run `02_prepare_labels.ipynb` first to generate `training/data/train.csv` etc.

In [ ]:
import pandas as pd
import os
from opensoundscape import CNN

DATA_DIR   = "../../training/data"
MODELS_DIR = "../../models"
os.makedirs(MODELS_DIR, exist_ok=True)

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), index_col=[0, 1, 2])
val   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"),   index_col=[0, 1, 2])
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"),  index_col=[0, 1, 2])

print(f"Classes: {list(train.columns)}")
print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

In [ ]:
# ── Build model ──────────────────────────────────────────────────────────────
model = CNN(
    architecture="resnet18",
    classes=list(train.columns),
    sample_duration=4.0,
)
print("Model built — architecture: resnet18")
print(f"Classes: {model.classes}")


In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
# Run config: 5 epochs, batch_size=32, num_workers=2, CPU.
# ~7-8h per epoch on CPU (~40h total). Use a GPU machine to cut this to minutes.

model.train(
    train,
    val,
    epochs=5,
    batch_size=32,
    save_path=os.path.join(MODELS_DIR, "orthoptera_checkpoints"),
    save_interval=1,
    num_workers=2,
)


In [ ]:
# ── Evaluate on held-out test set (6,459 clips) ──────────────────────────────
from sklearn.metrics import classification_report
import numpy as np

scores = model.predict(test)
y_true = test.values
y_pred = (scores.values > 0.5).astype(int)
print(classification_report(
    y_true, y_pred,
    target_names=list(train.columns),
    zero_division=0,
))


### Results — 5-epoch ResNet18 (CPU, May 2026)

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.96 | 0.61 | **0.75** | 2,530 |
| *Chorthippus brunneus brunneus* | 0.81 | 0.23 | **0.36** | 73 |
| *Pholidoptera griseoaptera* | 0.76 | 0.12 | **0.21** | 1,393 |
| *Leptophyes punctatissima* | 0.85 | 0.08 | **0.15** | 359 |
| *Gryllus campestris* | 0.94 | 0.06 | **0.12** | 929 |
| *Roeseliana roeselii* | 0.59 | 0.06 | **0.10** | 741 |
| *Pseudochorthippus parallelus* | 0.77 | 0.03 | **0.05** | 358 |
| *Omocestus viridulus* | 1.00 | 0.01 | **0.03** | 76 |

**Macro F1: 0.22 — precision is high (low false-positive rate) but recall is low.**

Pattern: the model is conservative after only 5 epochs — it fires confidently when it fires,
but misses most detections. *Tettigonia* performs well (most training data: 6,270 clips).

**To improve:**
- More epochs (20–30) — the main lever
- Lower `min_confidence` threshold in `config/settings.yaml` (try 0.3)
- GPU training would make 30 epochs feasible in <1 hour

**Threshold sweep verdict:** macro-F1 is flat across all thresholds (0.215–0.225). Lowering the threshold does not help — the model's score distribution is too compressed after only 5 epochs. More training epochs is the only meaningful lever.


In [ ]:
# ── Threshold sweep — find F1-maximising cutoff per species ──────────────────
import numpy as np
import pandas as pd

thresholds = np.arange(0.1, 0.91, 0.05)
results = []
for t in thresholds:
    y_pred_t = (scores.values > t).astype(int)
    from sklearn.metrics import f1_score
    macro_f1 = f1_score(y_true, y_pred_t, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred_t, average='micro', zero_division=0)
    results.append({'threshold': round(t, 2), 'macro_f1': round(macro_f1, 3), 'micro_f1': round(micro_f1, 3)})

sweep_df = pd.DataFrame(results)
best = sweep_df.loc[sweep_df['macro_f1'].idxmax()]
print(f"Best macro-F1 threshold: {best['threshold']}  (macro={best['macro_f1']}, micro={best['micro_f1']})")
print()
print(sweep_df.to_string(index=False))


Best macro-F1 threshold: 0.1  (macro=0.225, micro=0.443)

 threshold  macro_f1  micro_f1
      0.10     0.225     0.443
      0.15     0.225     0.443
      0.20     0.225     0.442
      0.25     0.225     0.443
      0.30     0.222     0.442
      0.35     0.222     0.441
      0.40     0.221     0.441
      0.45     0.221     0.441
      0.50     0.221     0.441
      0.55     0.221     0.441
      0.60     0.219     0.440
      0.65     0.219     0.440
      0.70     0.219     0.440
      0.75     0.219     0.440
      0.80     0.216     0.439
      0.85     0.216     0.439
      0.90     0.215     0.439


In [ ]:
# ── Save final model ──────────────────────────────────────────────────────────
# OpenSoundscape saves as a .model file (a torch pickle).
# This is the file you point insect.py at in BASE.

model_path = os.path.join(MODELS_DIR, "orthoptera_uk.model")
model.save(model_path)
print(f"Model saved: {model_path}")
print()
print("Next step: set 'model_path' in config/settings.yaml under 'insect:'")
print("and activate 'insect' in classifiers.active")